# 05 LoRA MVP Eval Colab

本 notebook 专门评测正式 LoRA MVP adapter。`04_train_lora_mvp_colab.ipynb` 负责训练；这里负责加载 `checkpoints/qwen3-asr-1.7b-lora-mvp/adapter/`，在固定 MVP 150 held-out test 上跑 LoRA always-on 推理、WER/CER、错误分析，并和 base baseline 指标对比。


In [ ]:
# 挂载 Google Drive。所有输入、checkpoint 和输出都放在 Drive 项目目录中。
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
# 安装最小依赖。qwen-asr 0.0.6 当前固定依赖 transformers==4.57.6 和 accelerate==1.12.0。
# 推理需要 peft 加载 adapter，需要 bitsandbytes 支持 4bit base。
# 当前评测不依赖 torchao。部分 Colab runtime 预装 torchao==0.10.0，
# 会导致 PEFT 加载 LoRA adapter 时触发版本不兼容，因此这里主动卸载。
%pip -q install --upgrade --upgrade-strategy only-if-needed qwen-asr==0.0.6 transformers==4.57.6 accelerate==1.12.0 peft==0.19.1 bitsandbytes huggingface_hub
%pip -q install pandas==2.2.2 requests==2.32.4
%pip -q uninstall -y torchao


In [ ]:
# 项目路径和评测参数。
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/qwen3-asr')
MANIFEST = PROJECT_DIR / 'data/jsonl/baseline_mvp_150.local.jsonl'
AUDIO_ROOT = PROJECT_DIR
ADAPTER_DIR = PROJECT_DIR / 'checkpoints/qwen3-asr-1.7b-lora-mvp/adapter'
BASE_METRICS = PROJECT_DIR / 'outputs/baseline_mvp_150/metrics.qwen3_asr_base.mvp_150.json'
OUTPUT_DIR = PROJECT_DIR / 'outputs/lora_mvp_eval'

MODEL_ID = 'Qwen/Qwen3-ASR-1.7B'
DTYPE = 'float16'
DEVICE_MAP = 'cuda:0'
QUANTIZATION = '4bit'
LANGUAGE = 'English'
MAX_NEW_TOKENS = 128
MAX_INFERENCE_BATCH_SIZE = 1

PRED_JSONL = OUTPUT_DIR / 'predictions.qwen3_asr_lora_mvp.mvp_150.jsonl'
MINI_MANIFEST = OUTPUT_DIR / 'baseline_mvp_150.mini_clean_degraded.jsonl'
MINI_PRED_JSONL = OUTPUT_DIR / 'predictions.qwen3_asr_lora_mvp.mini.jsonl'
SCORED_JSONL = OUTPUT_DIR / 'predictions.qwen3_asr_lora_mvp.mvp_150.scored.jsonl'
METRICS_JSON = OUTPUT_DIR / 'metrics.qwen3_asr_lora_mvp.mvp_150.json'
SCENARIO_CSV = OUTPUT_DIR / 'metrics_by_scenario.qwen3_asr_lora_mvp.mvp_150.csv'
ERROR_DIR = OUTPUT_DIR / 'error_analysis'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_DIR =', PROJECT_DIR)
print('MANIFEST =', MANIFEST)
print('ADAPTER_DIR =', ADAPTER_DIR)
print('OUTPUT_DIR =', OUTPUT_DIR)


In [ ]:
# 检查 GPU、manifest、adapter 和 baseline 指标是否存在。
# 缺任何一个都先停下来，避免跑到模型加载阶段才发现路径错了。
import json
import subprocess
from collections import Counter

subprocess.run(['nvidia-smi'], check=False)

required = [
    PROJECT_DIR / 'inference/qwen3_asr_lora_infer.py',
    PROJECT_DIR / 'evaluation/eval_wer.py',
    PROJECT_DIR / 'evaluation/analyze_errors.py',
    MANIFEST,
    ADAPTER_DIR / 'adapter_config.json',
    ADAPTER_DIR / 'adapter_model.safetensors',
    BASE_METRICS,
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError('缺少必要文件:\n' + '\n'.join(missing))

rows = [json.loads(line) for line in MANIFEST.read_text(encoding='utf-8').splitlines() if line.strip()]
counts = Counter(row.get('scenario', '') for row in rows)
missing_audio = []
for row in rows:
    audio = Path(row.get('audio') or row.get('audio_path') or '')
    resolved = audio if audio.is_absolute() else AUDIO_ROOT / audio
    if not resolved.exists():
        missing_audio.append(str(resolved))
print('manifest rows =', len(rows), 'scenario counts =', dict(counts), 'missing_audio =', len(missing_audio))
if missing_audio:
    print('\n'.join(missing_audio[:20]))
    raise FileNotFoundError(f'MVP 150 manifest 有缺失音频: {len(missing_audio)}')

clean_row = next((row for row in rows if row.get('scenario') == 'clean'), None)
degraded_row = next((row for row in rows if row.get('scenario') != 'clean'), None)
if clean_row is None or degraded_row is None:
    raise ValueError('mini manifest 需要至少 1 条 clean 和 1 条 degraded 样本')
MINI_MANIFEST.write_text(
    ''.join(json.dumps(row, ensure_ascii=False) + '\n' for row in [clean_row, degraded_row]),
    encoding='utf-8',
)
print('mini manifest =', MINI_MANIFEST, 'scenarios =', [clean_row.get('scenario'), degraded_row.get('scenario')])


In [ ]:
# 可选 Hugging Face 登录。
# 如果模型下载遇到权限或限流问题，在 Colab Secrets 里设置 HF_TOKEN 后重跑本 cell。
import os

try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
except Exception:
    token = None

if token:
    os.environ['HF_TOKEN'] = token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = token
    print('HF token detected from Colab Secrets.')
else:
    print('No HF token found in Colab Secrets. Public download will be used.')


In [ ]:
# 先跑 2 条 mini LoRA inference，确认 adapter 可以加载并转写。
import subprocess
import sys

cmd = [
    sys.executable,
    'inference/qwen3_asr_lora_infer.py',
    '--manifest', str(MINI_MANIFEST),
    '--output-jsonl', str(MINI_PRED_JSONL),
    '--adapter-dir', str(ADAPTER_DIR),
    '--audio-root', str(AUDIO_ROOT),
    '--model-id', MODEL_ID,
    '--dtype', DTYPE,
    '--device-map', DEVICE_MAP,
    '--quantization', QUANTIZATION,
    '--max-inference-batch-size', str(MAX_INFERENCE_BATCH_SIZE),
    '--max-new-tokens', str(MAX_NEW_TOKENS),
    '--language', LANGUAGE,
]
print('运行命令:')
print(' '.join(cmd))
proc = subprocess.run(cmd, cwd=PROJECT_DIR, text=True, capture_output=True)
print('returncode =', proc.returncode)
print('--- stdout tail ---')
print(proc.stdout[-4000:])
print('--- stderr tail ---')
print(proc.stderr[-4000:])
if proc.returncode != 0:
    raise RuntimeError('LoRA mini inference failed')


In [ ]:
# 预览 mini inference 输出。
mini_rows = [json.loads(line) for line in MINI_PRED_JSONL.read_text(encoding='utf-8').splitlines() if line.strip()]
for row in mini_rows:
    print('scenario =', row.get('scenario'), 'error =', row.get('error'))
    print('answer     :', row.get('answer'))
    print('prediction :', row.get('prediction'))
    print('-' * 80)


In [ ]:
# 跑完整 MVP 150 LoRA always-on inference。
cmd = [
    sys.executable,
    'inference/qwen3_asr_lora_infer.py',
    '--manifest', str(MANIFEST),
    '--output-jsonl', str(PRED_JSONL),
    '--adapter-dir', str(ADAPTER_DIR),
    '--audio-root', str(AUDIO_ROOT),
    '--model-id', MODEL_ID,
    '--dtype', DTYPE,
    '--device-map', DEVICE_MAP,
    '--quantization', QUANTIZATION,
    '--max-inference-batch-size', str(MAX_INFERENCE_BATCH_SIZE),
    '--max-new-tokens', str(MAX_NEW_TOKENS),
    '--language', LANGUAGE,
]
print('运行命令:')
print(' '.join(cmd))
proc = subprocess.run(cmd, cwd=PROJECT_DIR, text=True, capture_output=True)
print('returncode =', proc.returncode)
print('--- stdout tail ---')
print(proc.stdout[-6000:])
print('--- stderr tail ---')
print(proc.stderr[-6000:])
if proc.returncode != 0:
    raise RuntimeError('LoRA full inference failed')


In [ ]:
# 对 LoRA prediction JSONL 计算 WER/CER。
cmd = [
    sys.executable,
    'evaluation/eval_wer.py',
    '--predictions-jsonl', str(PRED_JSONL),
    '--scored-jsonl', str(SCORED_JSONL),
    '--metrics-json', str(METRICS_JSON),
    '--metrics-by-scenario-csv', str(SCENARIO_CSV),
]
print('运行命令:')
print(' '.join(cmd))
proc = subprocess.run(cmd, cwd=PROJECT_DIR, text=True, capture_output=True)
print('returncode =', proc.returncode)
print('--- stdout ---')
print(proc.stdout)
print('--- stderr ---')
print(proc.stderr)
if proc.returncode != 0:
    raise RuntimeError('LoRA WER/CER eval failed')


In [ ]:
# 运行错误分析，方便后续抽样看 LoRA 变好/变差的样本。
cmd = [
    sys.executable,
    'evaluation/analyze_errors.py',
    '--scored-jsonl', str(SCORED_JSONL),
    '--output-dir', str(ERROR_DIR),
]
print('运行命令:')
print(' '.join(cmd))
proc = subprocess.run(cmd, cwd=PROJECT_DIR, text=True, capture_output=True)
print('returncode =', proc.returncode)
print('--- stdout ---')
print(proc.stdout)
print('--- stderr ---')
print(proc.stderr)
if proc.returncode != 0:
    raise RuntimeError('LoRA error analysis failed')


In [ ]:
# 对比 base 与 LoRA 的 overall 和 scenario-level WER。
# 正数 delta 表示 LoRA WER 更高，负数 delta 表示 LoRA 改善。
import json
import pandas as pd

base = json.loads(BASE_METRICS.read_text(encoding='utf-8'))
lora = json.loads(METRICS_JSON.read_text(encoding='utf-8'))

def by_group(metrics):
    return {row['group']: row for row in metrics.get('by_scenario', [])}

base_groups = by_group(base)
lora_groups = by_group(lora)
scenarios = ['ALL', 'clean', 'noise', 'reverb', 'dropout', 'far_field']
rows_out = []

base_overall = base.get('overall', [{}])[0] if isinstance(base.get('overall'), list) else base.get('overall', {})
lora_overall = lora.get('overall', [{}])[0] if isinstance(lora.get('overall'), list) else lora.get('overall', {})
base_groups['ALL'] = base_overall
lora_groups['ALL'] = lora_overall

for scenario in scenarios:
    b = base_groups.get(scenario, {})
    v = lora_groups.get(scenario, {})
    base_wer = float(b.get('error_rate', 0.0) or 0.0)
    lora_wer = float(v.get('error_rate', 0.0) or 0.0)
    delta = lora_wer - base_wer
    relative = (delta / base_wer) if base_wer else None
    rows_out.append({
        'scenario': scenario,
        'base_wer': base_wer,
        'lora_wer': lora_wer,
        'delta_lora_minus_base': delta,
        'relative_delta': relative,
        'lora_empty_output_rate': v.get('empty_output_rate', 0.0),
    })

df = pd.DataFrame(rows_out)
display(df)

clean = df[df['scenario'] == 'clean'].iloc[0]
noise = df[df['scenario'] == 'noise'].iloc[0]
reverb = df[df['scenario'] == 'reverb'].iloc[0]
clean_relative = clean['relative_delta']
print('clean relative delta =', clean_relative)
print('noise relative delta =', noise['relative_delta'])
print('reverb relative delta =', reverb['relative_delta'])

if clean_relative is not None and clean_relative > 0.05:
    print('警告: clean regression 超过 5% 相对退化。')
if (noise['delta_lora_minus_base'] < 0) or (reverb['delta_lora_minus_base'] < 0):
    print('LoRA MVP 初步满足: noise 或 reverb 至少一个场景相对 base 改善。')
else:
    print('LoRA MVP 暂未满足: noise/reverb 都没有相对 base 改善，先不要进入 router。')


In [ ]:
# 列出本次评测产物。
for path in sorted(OUTPUT_DIR.rglob('*')):
    if path.is_file():
        print(path.relative_to(PROJECT_DIR), path.stat().st_size)
